# 🧠 Personal Knowledge Guardian: Colab Ollama Server

This notebook runs **Ollama** on Google Colab's free GPU tier, caching models onto Google Drive to avoid downloads on every container boot, and tunneling the Ollama port to a public HTTPS endpoint via **Ngrok**. This serves as a free alternative to Modal.com.

### Instructions:
1. Get a free Ngrok auth token from [Ngrok Dashboard](https://dashboard.ngrok.com/auth).
2. Add your token in the cell below (or store it as a Colab secret under `NGROK_AUTH_TOKEN`).
3. Run all cells (`Ctrl + F9`).

In [ ]:
# --- Step 1: Mount Google Drive (Optional) ---
import os

use_drive = False
try:
    from google.colab import drive
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
    use_drive = True
    print("✓ Google Drive mounted successfully.")
except Exception as e:
    print(f"⚠️ Google Drive mount was unsuccessful or skipped ({e}).")
    print("Defaulting to local runtime storage (models will not persist across runtime resets).")

# Set up storage directory paths based on Google Drive availability
if use_drive:
    models_dir = '/content/drive/MyDrive/ollama_models'
    data_dir = '/content/drive/MyDrive/ollama_data'
else:
    models_dir = '/content/ollama_models'
    data_dir = '/content/ollama_data'

os.makedirs(models_dir, exist_ok=True)
os.makedirs(data_dir, exist_ok=True)

# Direct Ollama to read/write models from/to the selected storage path
os.environ['OLLAMA_MODELS'] = models_dir
print(f"✓ Ollama models path configured at: {models_dir}")

In [ ]:
# --- Step 2: Install Dependencies (including zstd for Ollama extraction) ---
!apt-get update
!apt-get install -y curl build-essential unzip zstd

# Install Ollama CLI & Daemon
!curl -fsSL https://ollama.com/install.sh | sh

# Install langchain client wrappers and support libraries
!pip install langchain-ollama pyngrok flask flask-cors requests

In [ ]:
# --- Step 3: Configure pyngrok ---
# pyngrok will automatically download and manage the ngrok binary under the hood
from pyngrok import ngrok
print("✓ pyngrok library imported.")

In [ ]:
# --- Step 4: Configure Ngrok Authorization ---
try:
    from google.colab import userdata
    NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
    print("✓ Ngrok token retrieved from Colab secrets.")
except Exception:
    # Fallback to write-in token if Colab secret is not set
    NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN"

if not NGROK_AUTH_TOKEN or NGROK_AUTH_TOKEN == "YOUR_NGROK_AUTH_TOKEN":
    raise ValueError("❌ Please specify a valid NGROK_AUTH_TOKEN in this cell or in Colab Secrets.")

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("✓ Ngrok auth token configured successfully.")

In [ ]:
# --- Step 5: Boot Ollama Server Daemon ---
import subprocess
import time

print("Starting Ollama in background...")
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for daemon to initialize binding
time.sleep(10)

In [ ]:
# --- Step 6: Pull LLM Model Weights ---
import os

model_name = "gemma2:2b"  # Choose from 'llama3.1', 'gemma2:2b', 'qwen2.5:3b', 'llama3.2:3b'
print(f"⬇️ Ensuring model '{model_name}' is loaded (Ollama will skip download if already cached)...")
!ollama pull {model_name}
print("✅ Model check completed!")

In [ ]:
# --- Step 7: Launch Ngrok Tunnel with Host Rewriting ---
import os
from pyngrok import ngrok

print("Exposing Ollama on port 11434 via pyngrok...")
public_url = ""

try:
    # Disconnect active tunnels and kill any existing local ngrok processes to prevent ERR_NGROK_334
    print("Cleaning up existing tunnels...")
    tunnels = ngrok.get_tunnels()
    for t in tunnels:
        ngrok.disconnect(t.public_url)
    ngrok.kill()
except Exception as cleanup_err:
    print(f"(Note: No active tunnels to clean up: {cleanup_err})")

try:
    # Rewrite Host header to localhost to bypass Ollama 403 Security Policy
    tunnel = ngrok.connect(11434, "http", host_header="rewrite")
    public_url = tunnel.public_url
    print(f"✓ Ngrok tunnel established successfully!")
    print(f"📡 Public URL: {public_url}")
    print(f"📡 Ollama OpenAI-Compatible endpoint: {public_url}/v1")
    
    # Save endpoint to a text file for backend auto-discovery
    endpoint_txt_path = os.path.join(data_dir, "endpoint_url.txt")
    with open(endpoint_txt_path, 'w', encoding='utf-8') as f:
        f.write(f"{public_url}/v1")
    print(f"💾 Saved endpoint to: {endpoint_txt_path}")
except Exception as e:
    print(f"❌ Error establishing tunnel: {e}")
    print("Please review your Ngrok credentials and status.")

In [ ]:
# --- Step 8: Verify Model Response ---
from langchain_ollama import OllamaLLM

if public_url:
    print("Initializing LangChain tester...")
    llm = OllamaLLM(
        base_url=public_url,
        model=model_name,
        temperature=0.3,
        max_tokens=1024
    )

    query = "Explain what an attention mechanism is in transformers in 2 sentences."
    print(f"Query: {query}")
    try:
        response = llm.invoke(query)
        print("\n🔬 Test Response:")
        print("=" * 60)
        print(response)
        print("=" * 60)
    except Exception as e:
        print(f"❌ Error querying LLM: {e}")
else:
    print("⚠️ Skipping verification since tunnel was not established.")

In [ ]:
# --- Step 9: Keep Server Session Alive ---
import time

if public_url:
    print(f"🚀 Ollama server is running and listening at: {public_url}/v1")
    print("⏳ Keeping the Colab session alive. Press Ctrl+C to abort...")

    try:
        while True:
            time.sleep(60)
    except KeyboardInterrupt:
        print("🛑 Terminating Ngrok tunnel...")
        ngrok.disconnect(public_url)
        print("✓ Server stopped.")
else:
    print("⚠️ Server is not running because of tunnel error.")